# Flash Flash Revolution Silver Layer Data Transformation

This notebook transforms raw Flash Flash Revolution game data from bronze into curated, analytics-ready silver tables. It supports **incremental processing** by detecting changed songs via `swf_version` tracking, reducing compute costs and execution time.

## Pipeline Overview

**Bronze → Silver Transformation:**
1. Detect new, updated, and deleted songs using `swf_version`
2. Transform playlist metadata (genre mapping, time conversion, boolean flags)
3. Explode nested chart arrays into note-level records
4. Apply one-hot encoding for note orientations
5. Upsert transformed data using MERGE (playlist) or DELETE + INSERT (notes)

**Source Tables:**
- `acubed.ffr.bronze__playlist` - Raw song metadata with encoded genre IDs
- `acubed.ffr.bronze__charts` - Nested arrays containing note sequences
- `acubed.ffr.bronze__songlist` - Song versions for change detection

**Target Tables:**
- `acubed.ffr.silver__playlist` - Enriched song metadata with human-readable labels
- `acubed.ffr.silver__notes` - Exploded note-level data with orientation encoding

---

## Incremental Processing Logic

**Change Detection:**
1. Join bronze_songlist with silver_playlist to find:
   - **New songs**: In bronze but not in silver
   - **Updated songs**: `swf_version` differs between bronze and silver
   - **Deleted songs**: In silver but not in bronze
2. Only process changed songs (skip if no changes detected)
3. Exit early if no work required

**Version Tracking:**
- `swf_version` from bronze__songlist identifies each song's content state
- Stored in silver tables to enable future change detection
- Upstream chart updates trigger new versions

**Update Strategy:**
- **Playlist**: MERGE operation (upsert on song_id)
- **Notes**: DELETE existing notes for changed songs, then INSERT new notes
- **Deletes**: Remove songs no longer in bronze layer

---

## Playlist Transformations

**Genre Mapping:**
- Decode numeric genre IDs to human-readable names
- Mapping: `{1: "Dance", 2: "Chill", 3: "Techno", ...}`
- Ensures consistent genre labels across analyses

**Time Conversion:**
- Parse Unix timestamps to readable datetime format
- Use `from_unixtime()` for release date fields

**Boolean Flags:**
- Convert numeric flags (0/1) to proper boolean types
- Fields: `secret`, `tutorial`, `preloaded`

**Schema Enrichment:**
- Add `swf_version` for change tracking (if missing)
- Preserve all original fields from bronze layer
- Convert encoded values to analysis-ready formats

---

## Notes Transformations

**Array Explosion:**
- Use `posexplode()` to convert nested arrays into rows
- Each note becomes a separate record with position index
- Preserves note order via position field (`note_id`)

**Orientation Encoding:**
- Parse 4-character orientation strings (e.g., "1001", "0100")
- Each position represents an arrow direction:
  - Position 0: Left arrow
  - Position 1: Down arrow
  - Position 2: Up arrow
  - Position 3: Right arrow
- Values: "1" = active, "0" = inactive

**Color Mapping:**
- Decode numeric color IDs to color names
- Mapping: `{0: "red", 1: "blue", 2: "green", ...}`
- Supports visual analysis and gameplay mechanics

**Timestamp Calculation:**
- Convert frame counts to seconds
- Formula: `timestamp = framers / 30.0` (30 FPS)
- Enables temporal analysis and density calculations

---

## Data Quality Improvements

**Consistency:**
- Human-readable labels replace numeric codes
- Standardized data types (boolean for flags, datetime for dates)
- Explicit schema definitions

**Completeness:**
- All songs have version tracking
- All notes include orientation, color, and timing information
- No orphaned records (referential integrity maintained)

**Analysis-Ready:**
- Exploded structure enables note-level aggregations
- Proper data types support filtering and calculations
- Indexed columns (song_id, note_id) optimize joins

---

## Table Schemas

**silver__playlist:**
- `song_id` (bigint): Unique song identifier
- `name` (string): Song title
- `difficulty` (int): Difficulty rating
- `genre` (string): Human-readable genre name
- `release_date` (timestamp): When song was added to game
- `secret` (boolean): Whether song is hidden
- `tutorial` (boolean): Whether song is a tutorial
- `preloaded` (boolean): Whether song loads at game start
- `swf_version` (long): Version tracking for incremental updates
- Additional metadata fields preserved from bronze

**silver__notes:**
- `song_id` (bigint): Foreign key to playlist
- `note_id` (int): Note sequence number (0-indexed)
- `framers` (int): Frame count from source data
- `orientation` (string): 4-character arrow pattern (e.g., "1001")
- `color` (string): Human-readable color name
- `timestamp` (double): Note timing in seconds (framers/30)

In [0]:
from pyspark.sql.functions import (
    when, 
    col, 
    split, 
    from_unixtime, 
    posexplode, 
    max as spark_max, 
    lit
)
from delta.tables import DeltaTable

In [0]:
# Configuration: Automatic Processing Mode
# Automatically determines whether to do full refresh or incremental processing
# - Full refresh (process_all=True): When table doesn't exist (first run)
# - Incremental (process_all=False): When table exists (subsequent runs)

silver_table = "acubed.ffr.silver__playlist"
process_all = not spark.catalog.tableExists(silver_table)

if process_all:
    print(f"ℹ Auto-detected: Full refresh mode (table does not exist)")
else:
    print(f"ℹ Auto-detected: Incremental mode (table exists)")

print(f"✓ Configuration loaded: process_all = {process_all}")

In [0]:
# Identify songs that need to be processed (new, changed, or deleted)
# Integrates with bronze layer's swf_version tracking to detect updates

silver_table = "acubed.ffr.silver__playlist"

if spark.catalog.tableExists(silver_table) and not process_all:
    # Silver table exists - check for new songs, updated songs, and deleted songs
    bronze_song_ids = spark.table("acubed.ffr.bronze__playlist").select(
        col("level").cast("bigint").alias("song_id")  # Cast to bigint to match bronze__charts
    ).distinct()
    
    # Get bronze songlist with swf_version for change detection
    bronze_songlist = spark.table("acubed.ffr.bronze__songlist").select(
        col("id").alias("song_id"),
        col("swf_version").alias("bronze_swf_version")
    )
    
    silver_song_ids = spark.table(silver_table).select("song_id").distinct()
    
    # Find new songs (in bronze but not in silver)
    new_songs = bronze_song_ids.join(
        silver_song_ids,
        on="song_id",
        how="left_anti"
    )
    
    num_new = new_songs.count()
    
    # Find deleted songs (in silver but not in bronze)
    deleted_songs = silver_song_ids.join(
        bronze_song_ids,
        on="song_id",
        how="left_anti"
    )
    
    num_deleted = deleted_songs.count()
    
    # Find updated songs (swf_version changed in bronze vs silver)
    # Check if silver table has swf_version column
    silver_columns = spark.table(silver_table).columns
    
    if "swf_version" in silver_columns:
        # Silver has swf_version - compare with bronze
        silver_versions = spark.table(silver_table).select(
            col("song_id"),
            col("swf_version").alias("silver_swf_version")
        )
        
        updated_songs = bronze_songlist.join(
            silver_versions,
            on="song_id",
            how="inner"
        ).filter(
            col("bronze_swf_version") != col("silver_swf_version")
        ).select("song_id")
        
        num_updated = updated_songs.count()
    else:
        # Silver doesn't have swf_version yet - treat all existing songs as needing update
        print("⚠ Silver table missing swf_version column - will add it during transformation")
        updated_songs = silver_song_ids
        num_updated = updated_songs.count()
    
    # Handle deletions first
    if num_deleted > 0:
        print(f"Found {num_deleted} songs to delete from silver tables")
        
        # Get list of deleted song IDs to delete
        deleted_ids = [row['song_id'] for row in deleted_songs.collect()]
        deleted_ids_str = ','.join([str(id) for id in deleted_ids])
        
        # Delete from playlist table using MERGE
        DeltaTable.forName(spark, "acubed.ffr.silver__playlist").alias("target").merge(
            deleted_songs.alias("source"),
            "target.song_id = source.song_id"
        ).whenMatchedDelete().execute()
        
        # Delete from notes table using SQL DELETE (more reliable for type mismatches)
        if spark.catalog.tableExists("acubed.ffr.silver__notes"):
            spark.sql(f"DELETE FROM acubed.ffr.silver__notes WHERE song_id IN ({deleted_ids_str})")
        
        print(f"✓ Deleted {num_deleted} songs from silver tables")
    
    # Combine new and updated songs for processing
    print(f"\n{'='*60}")
    print(f"CHANGE DETECTION SUMMARY")
    print(f"{'='*60}")
    print(f"New songs: {num_new}")
    print(f"Updated songs (swf_version changed): {num_updated}")
    print(f"Deleted songs: {num_deleted}")
    print(f"Total songs to process: {num_new + num_updated}")
    print(f"{'='*60}")
    
    if num_new > 0 or num_updated > 0:
        if num_new > 0 and num_updated > 0:
            # Union both new and updated songs
            changed_songs_df = new_songs.union(updated_songs)
        elif num_new > 0:
            changed_songs_df = new_songs
        else:
            changed_songs_df = updated_songs
        
        # CRITICAL FIX: Materialize the changed song IDs immediately as a list
        # This prevents the DataFrame from becoming stale after silver table writes
        # Without this, the lazy DataFrame re-executes after Cell 5 writes to silver,
        # and the left_anti join returns 0 rows since songs are now in silver
        changed_song_id_list = [row['song_id'] for row in changed_songs_df.collect()]
        
        # Recreate DataFrame from materialized list
        changed_song_ids = spark.createDataFrame(
            [(song_id,) for song_id in changed_song_id_list],
            ["song_id"]
        )
        
        print(f"\n✓ Processing {len(changed_song_id_list)} songs (new + updated)")
        print(f"   Song IDs: {changed_song_id_list}")
    else:
        # No changes detected - set to None
        print("\nℹ No changes detected - transformation will be skipped")
        changed_song_ids = None
else:
    if process_all:
        print("ℹ Force full refresh mode enabled (process_all=True)")
    else:
        print("ℹ Silver table doesn't exist - performing initial load of all songs")
    changed_song_ids = None  # Set to None for full refresh

In [0]:
# Guard: Skip if no songs to process AND table already exists
if not process_all and changed_song_ids is None and spark.catalog.tableExists("acubed.ffr.silver__playlist"):
    print("ℹ No songs to process - skipping playlist transformation")
    print("✓ Notebook will complete successfully (idempotent run)")
    
    # Create empty DataFrame with correct schema for downstream compatibility
    df_transformed = spark.createDataFrame([], schema="""
        genre string,
        name string,
        difficulty int,
        style int,
        song_length int,
        song_id int,
        num_arrows int,
        preview_id string,
        release_date timestamp,
        author string,
        stepauthor array<string>,
        song_rating double,
        explicit_flag boolean,
        legacy_flag boolean,
        unranked_flag boolean,
        credits int,
        price int,
        swf_version string
    """)
else:
    # Load bronze table and filter to changed songs if doing incremental processing
    df = spark.table("acubed.ffr.bronze__playlist")

    if not process_all and changed_song_ids is not None:
        # Filter to only changed songs for incremental processing
        df = df.join(changed_song_ids, df.level == changed_song_ids.song_id, "inner").drop(changed_song_ids.song_id)
        print(f"Processing {df.count()} changed songs")

    # Join with bronze__songlist to get swf_version for change tracking
    bronze_songlist = spark.table("acubed.ffr.bronze__songlist").select(
        col("id").alias("song_id_join"),
        col("swf_version")
    )

    df = df.join(
        bronze_songlist,
        df.level == bronze_songlist.song_id_join,
        "left"
    ).drop("song_id_join")

    # Identify o_ columns for boolean transformation
    o_columns = [c for c in df.columns if c.startswith("o_")]

    # Build boolean transformations for all o_ columns at once
    o_column_transforms = {
        c: when(col(c) == 1, True).otherwise(False) for c in o_columns
    }

    # Apply all transformations in minimal steps
    df_transformed = df.withColumn(
        "genre",
        when(col("genre") == 1, "Dance")
        .when(col("genre") == 2, "Dance 2")
        .when(col("genre") == 3, "Funk")
        .when(col("genre") == 4, "Arcade")
        .when(col("genre") == 5, "Rock")
        .when(col("genre") == 6, "Classical")
        .when(col("genre") == 7, "Misc")
        .when(col("genre") == 8, "Secret")
        .when(col("genre") == 9, "Purchased")
        .when(col("genre") == 10, "Token")
        .when(col("genre") == 11, "Hip-Hop")
        .when(col("genre") == 12, "Skill Token")
        .when(col("genre") == 13, "Legacy")
        .otherwise(None)
    ).withColumn(
        "time",
        split(col("time"), ":")[0].cast("int") * 60 + split(col("time"), ":")[1].cast("int")
    ).withColumn(
        "stepauthor",
        split(col("stepauthor"), " & ")
    ).withColumn(
        "releasedate",
        from_unixtime(col("releasedate")).cast("timestamp")
    ).withColumns(
        o_column_transforms
    ).fillna(
        0, subset=["credits", "price"]
    ).withColumnsRenamed({
        "level": "song_id", 
        "time": "song_length",
        "previewhash": "preview_id", 
        "releasedate": "release_date",
        "arrows": "num_arrows",
        **{c: c.replace("o_", "") + "_flag" for c in o_columns}
    }).withColumns({
        "song_id": col("song_id").cast("int"),
        "num_arrows": col("num_arrows").cast("int"),
        "credits": col("credits").cast("int"),
        "price": col("price").cast("int")
    }).select(
        'genre', 'name', 'difficulty', 'style', 'song_length', 'song_id', 
        'num_arrows', 'preview_id', 'release_date', 'author', 'stepauthor', 
        'song_rating', 'explicit_flag', 'legacy_flag', 'unranked_flag', 
        'credits', 'price', 'swf_version'
    )

display(df_transformed)

In [0]:
# Upsert the transformed data to the silver table using MERGE
table_name = "acubed.ffr.silver__playlist"

# Capture count BEFORE merge (DataFrames are lazy, count after merge may be inaccurate)
rows_to_merge = df_transformed.count()

if rows_to_merge == 0:
    print("ℹ No rows to insert - skipping table update (idempotent run)")
else:
    if spark.catalog.tableExists(table_name):
        # Table exists - use MERGE for incremental updates
        
        # Check if swf_version column exists, if not add it
        existing_columns = [col.name for col in spark.table(table_name).schema]
        if 'swf_version' not in existing_columns:
            print("Adding swf_version column to silver table...")
            spark.sql(f"ALTER TABLE {table_name} ADD COLUMN swf_version STRING")
        
        delta_table = DeltaTable.forName(spark, table_name)
        
        delta_table.alias("target").merge(
            df_transformed.alias("source"),
            "target.song_id = source.song_id"
        ).whenMatchedUpdateAll(
        ).whenNotMatchedInsertAll(
        ).execute()
        
        print(f"Merged {rows_to_merge} rows into {table_name}")
    else:
        # Table doesn't exist - create it
        df_transformed.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
        print(f"Created {table_name} with {rows_to_merge} rows")

In [0]:
# Guard: Skip if no songs to process AND table already exists
if not process_all and changed_song_ids is None and spark.catalog.tableExists("acubed.ffr.silver__notes"):
    print("ℹ No songs to process - skipping notes transformation")
    print("✓ Notebook will complete successfully (idempotent run)")
    
    # Create empty DataFrame with correct schema for downstream compatibility
    df_notes_transformed = spark.createDataFrame([], schema="""
        song_id bigint,
        note_id int,
        framers int,
        orientation string,
        color string,
        timestamp double
    """)
else:
    # Load bronze charts table and filter to changed songs if doing incremental processing
    df_charts = spark.table("acubed.ffr.bronze__charts")

    if not process_all and changed_song_ids is not None:
        # Filter to only changed songs for incremental processing
        df_charts = df_charts.join(changed_song_ids, "song_id", "inner")
        num_songs = changed_song_ids.count()
        print(f"Processing charts for {num_songs} changed songs")
        
        # Check if there are any charts to process
        if df_charts.count() == 0:
            print("ℹ No charts found for changed songs - skipping notes transformation")
            
            # Create empty DataFrame with correct schema
            df_notes_transformed = spark.createDataFrame([], schema="""
                song_id bigint,
                note_id int,
                framers int,
                orientation string,
                color string,
                timestamp double
            """)
        else:
            # Explode the chart array with position (note_id starting at 0)
            df_exploded = df_charts.select(
                col("song_id"),
                posexplode(col("chart")).alias("note_index", "note")
            )

            # Extract the 4 elements from each note array and create note_id starting at 1
            df_notes = df_exploded.select(
                col("song_id"),
                (col("note_index") + 1).alias("note_id"),
                col("note")[0].cast("int").alias("framers"),
                col("note")[1].cast("int").alias("orientation"),
                col("note")[2].cast("int").alias("color"),
                (col("note")[3].cast("int") / 1000).alias("timestamp")
            )

            # Check if df_notes is empty
            if df_notes.count() == 0:
                print("ℹ No notes to process - skipping notes transformation")
                
                df_notes_transformed = spark.createDataFrame([], schema="""
                    song_id bigint,
                    note_id int,
                    framers int,
                    orientation string,
                    color string,
                    timestamp double
                """)
            else:
                # Dynamically determine the number of distinct orientation values
                max_orientation = df_notes.select(spark_max(col("orientation"))).collect()[0][0]

                # Handle case where max_orientation is None (empty dataframe)
                if max_orientation is None:
                    print("ℹ No orientation values found - skipping notes transformation")
                    
                    df_notes_transformed = spark.createDataFrame([], schema="""
                        song_id bigint,
                        note_id int,
                        framers int,
                        orientation string,
                        color string,
                        timestamp double
                    """)
                else:
                    num_orientations = max_orientation + 1  # 0-indexed, so add 1

                    print(f"Detected {num_orientations} distinct orientation values (0 to {max_orientation})")

                    # Create orientation mapping: each value i maps to a one-hot encoded string
                    # For 4 values: 0->'1000', 1->'0100', 2->'0010', 3->'0001'
                    # For 6 values: 0->'100000', 1->'010000', etc.
                    orientation_expr = None
                    for i in range(num_orientations):
                        # Create binary string with '1' at position i, '0' elsewhere
                        binary_str = ''.join(['1' if j == i else '0' for j in range(num_orientations)])
                        
                        if orientation_expr is None:
                            orientation_expr = when(col("orientation") == i, lit(binary_str))
                        else:
                            orientation_expr = orientation_expr.when(col("orientation") == i, lit(binary_str))

                    # Apply the orientation transformation
                    df_notes_transformed = df_notes.withColumn(
                        "orientation",
                        orientation_expr.otherwise(None)
                    )

                    # Map color integers to color names
                    df_notes_transformed = df_notes_transformed.withColumn(
                        "color",
                        when(col("color") == 0, "blue")
                        .when(col("color") == 1, "red")
                        .when(col("color") == 2, "purple")
                        .when(col("color") == 3, "yellow")
                        .when(col("color") == 4, "pink")
                        .when(col("color") == 5, "orange")
                        .when(col("color") == 6, "cyan")
                        .when(col("color") == 7, "green")
                        .when(col("color") == 8, "white")
                        .otherwise(None)
                    )
    else:
        # Full refresh - process all songs
        # Explode the chart array with position (note_id starting at 0)
        df_exploded = df_charts.select(
            col("song_id"),
            posexplode(col("chart")).alias("note_index", "note")
        )

        # Extract the 4 elements from each note array and create note_id starting at 1
        df_notes = df_exploded.select(
            col("song_id"),
            (col("note_index") + 1).alias("note_id"),
            col("note")[0].cast("int").alias("framers"),
            col("note")[1].cast("int").alias("orientation"),
            col("note")[2].cast("int").alias("color"),
            (col("note")[3].cast("int") / 1000).alias("timestamp")
        )

        # Dynamically determine the number of distinct orientation values
        max_orientation = df_notes.select(spark_max(col("orientation"))).collect()[0][0]
        num_orientations = max_orientation + 1  # 0-indexed, so add 1

        print(f"Detected {num_orientations} distinct orientation values (0 to {max_orientation})")

        # Create orientation mapping
        orientation_expr = None
        for i in range(num_orientations):
            binary_str = ''.join(['1' if j == i else '0' for j in range(num_orientations)])
            
            if orientation_expr is None:
                orientation_expr = when(col("orientation") == i, lit(binary_str))
            else:
                orientation_expr = orientation_expr.when(col("orientation") == i, lit(binary_str))

        # Apply the orientation transformation
        df_notes_transformed = df_notes.withColumn(
            "orientation",
            orientation_expr.otherwise(None)
        )

        # Map color integers to color names
        df_notes_transformed = df_notes_transformed.withColumn(
            "color",
            when(col("color") == 0, "blue")
            .when(col("color") == 1, "red")
            .when(col("color") == 2, "purple")
            .when(col("color") == 3, "yellow")
            .when(col("color") == 4, "pink")
            .when(col("color") == 5, "orange")
            .when(col("color") == 6, "cyan")
            .when(col("color") == 7, "green")
            .when(col("color") == 8, "white")
            .otherwise(None)
        )

display(df_notes_transformed)

In [0]:
# Upsert the transformed notes data to the silver table using MERGE
table_name = "acubed.ffr.silver__notes"

# Capture counts BEFORE operations (DataFrames are lazy, counts after operations may be inaccurate)
rows_to_insert = df_notes_transformed.count()

if rows_to_insert == 0:
    print("ℹ No rows to insert - skipping table update (idempotent run)")
else:
    if spark.catalog.tableExists(table_name):
        # Table exists - use MERGE for incremental updates
        # For notes, we need to delete old notes for changed songs and insert new ones
        delta_table = DeltaTable.forName(spark, table_name)
        
        # Get list of song_ids being updated
        updated_song_ids = df_notes_transformed.select("song_id").distinct()
        num_songs = updated_song_ids.count()
        
        if num_songs > 0:
            # Delete old notes for songs being updated
            delta_table.alias("target").merge(
                updated_song_ids.alias("source"),
                "target.song_id = source.song_id"
            ).whenMatchedDelete().execute()
            
            # Insert new notes
            df_notes_transformed.write.mode("append").saveAsTable(table_name)
            
            print(f"Updated notes for {num_songs} songs ({rows_to_insert} rows) in {table_name}")
        else:
            print("ℹ No songs to update - skipping table operations")
    else:
        # Table doesn't exist - create it
        df_notes_transformed.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
        print(f"Created {table_name} with {rows_to_insert} rows")

In [0]:
# # Delete song_id = 4104 from all tables to test incremental logic

# song_id_to_delete = 4104

# print(f"Deleting song_id = {song_id_to_delete} from all tables...\n")

# # Bronze layer tables
# tables_to_clean = [
#     "acubed.ffr.bronze__songlist",
#     "acubed.ffr.bronze__playlist",
#     "acubed.ffr.bronze__charts",
#     "acubed.ffr.silver__playlist",
#     "acubed.ffr.silver__notes"
# ]

# for table in tables_to_clean:
#     if spark.catalog.tableExists(table):
#         # Get count before deletion
#         before_count = spark.table(table).count()
        
#         # Determine the song_id column name for each table
#         if "songlist" in table:
#             id_column = "id"
#         elif "playlist" in table:
#             id_column = "level" if "bronze" in table else "song_id"
#         else:
#             id_column = "song_id"
        
#         # Delete the song
#         spark.sql(f"DELETE FROM {table} WHERE {id_column} = {song_id_to_delete}")
        
#         # Get count after deletion
#         after_count = spark.table(table).count()
#         deleted_count = before_count - after_count
        
#         print(f"✓ {table}: Deleted {deleted_count} rows (before: {before_count}, after: {after_count})")
#     else:
#         print(f"⚠ {table}: Table doesn't exist")

# print(f"\n✓ Successfully deleted song_id = {song_id_to_delete} from all tables")
# print(f"\nNow you can run the bronze ingestion notebook to test incremental detection!")